We first begin by joining together the weather and flight datasets.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [3]:
con = duckdb.connect('flight_data.duckdb')
con.execute("ATTACH 'weather_data.duckdb' AS weather_db")

In [4]:
query = """
SELECT 
    f.MONTH, 
    f.DAY_OF_MONTH, 
    f.DISTANCE,
    CAST(f.CRS_DEP_TIME / 100 AS BIGINT) AS DEP_HOUR,
    
    -- All Weather Features included with logical imputations
    COALESCE(w.WIND_SPEED, 0) AS WIND_SPEED,
    COALESCE(w.WIND_GUST, 0) AS WIND_GUST,
    COALESCE(w.WIND_ANGLE, 0) AS WIND_ANGLE,
    COALESCE(w.VISIBILITY, 10) AS VISIBILITY,
    COALESCE(w.AIR_TEMP, 20) AS AIR_TEMP,
    COALESCE(w.DEW_POINT_TEMP, w.AIR_TEMP) AS DEW_POINT_TEMP, -- Default to air temp if missing
    COALESCE(w.PRESSURE, 1013.25) AS PRESSURE, -- Standard atmospheric pressure
    COALESCE(w.PRECIPITATION, 0) AS PRECIPITATION,
    COALESCE(w.CEILING_HEIGHT, 99999) AS CEILING_HEIGHT, -- 99999 implies infinite/clear ceiling
    
    -- Boolean flags
    COALESCE(w.HAS_THUNDERSTORM, 0) AS HAS_THUNDERSTORM,
    COALESCE(w.HAS_GUST, 0) AS HAS_GUST,
    COALESCE(w.HAS_CLOUDS, 0) AS HAS_CLOUDS,
    
    f.ARRIVAL_DELAYED AS target
FROM flights_cleaned f
LEFT JOIN weather_db.weather_cleaned w
    ON f.ORIGIN = w.STATION
    AND f.YEAR = w.YEAR
    AND f.MONTH = w.MONTH
    AND f.DAY_OF_MONTH = w.DAY
    AND CAST(f.CRS_DEP_TIME / 100 AS BIGINT) = w.HOUR
WHERE f.ARRIVAL_DELAYED IS NOT NULL
  AND f.CANCELLED = 0 
"""

df = con.execute(query).fetchdf()
print(f"Dataset shape with all features: {df.shape}")

CatalogException: Catalog Error: Table with name flights_cleaned does not exist!
Did you mean "sqlite_schema"?

LINE 25: FROM flights_cleaned f
              ^

Building the basic model.

In [ ]:
# Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target'].values

# Split into training and validation sets (80/20 split)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
class FlightDataset(Dataset):
    def __init__(self, features, labels):
        # Convert to PyTorch tensors (Float32 for features, Float32 for BCEWithLogitsLoss)
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # Reshape to [N, 1]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create Datasets
train_dataset = FlightDataset(X_train_scaled, y_train)
val_dataset = FlightDataset(X_val_scaled, y_val)

# Create DataLoaders (Batch size of 1024 is good for simple models and fast GPUs)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        # Single linear layer (Input -> 1 Output)
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        # Output raw logits
        return self.linear(x)

# Initialize the model
input_dim = X_train_scaled.shape[1]
model = LogisticRegression(input_dim)

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

In [ ]:
# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        
    # Calculate average training loss
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            
            # Apply sigmoid to get probabilities, then round to 0 or 1
            predicted = torch.sigmoid(outputs).round()
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}")